<a href="https://colab.research.google.com/github/csu-techhub/QOS-Labs/blob/main/Lab6.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>


# Lab 6 — H₂ Bond-Length Curve, Noise, and Optional Hardware
**Quantum Optimization and Simulation — VQE Laboratory Series**

Connect VQE's electronic inner loop to geometry optimization.

> Qiskit displays measured bitstrings as `q_(n-1)...q_0`. When orbital labels are written in the order `q0, q1, ...`, this notebook explicitly notes the convention.

## Learning objectives
- Distinguish electronic optimization at fixed bond length from geometry optimization.
- Construct a simple H₂ potential-energy curve from supplied Hamiltonians.
- Compare exact, shot-based, and noisy estimates.
- Optionally transpile/run one point on IBM hardware.

In [ ]:
# Run once in a fresh Google Colab session.
%pip -q install "pylatexenc" "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8" "qiskit-ibm-runtime~=0.46"

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

SEED = 123
SHOTS = 4096
# Helper functions
def run_counts(qc, shots=2048, backend=None):

    # Ideal simulation
    if backend is None:
        sim = AerSimulator()
    else:
        sim = backend

    # Transpile for the selected backend
    tqc = transpile(qc, sim, optimization_level=1)

    result = sim.run(
        tqc,
        shots=shots
    ).result()

    return result.get_counts()

def q0_first(qiskit_bits):
    return qiskit_bits.replace(" ", "")[::-1]

In [ ]:
def ansatz(theta):
    qc = QuantumCircuit(2)

    # Prepare |01>
    qc.x(1)

    # ------------------------------------------------
    # RYX(theta) = exp[-i theta/2 (Y0 X1)]
    # ------------------------------------------------
    qc.sdg(0)
    qc.h(0)
    qc.h(1)

    # ZZ rotation
    qc.cx(0, 1)
    qc.rz(theta, 1)
    qc.cx(0, 1)

    # Undo basis change
    qc.h(0)
    qc.s(0)
    qc.h(1)

    # ------------------------------------------------
    # RXY(-theta) = exp[+i theta/2 (X0 Y1)]
    # ------------------------------------------------

    qc.h(0)
    qc.sdg(1)
    qc.h(1)

    # ZZ rotation with angle -theta
    qc.cx(0, 1)
    qc.rz(-theta, 1)
    qc.cx(0, 1)

    # Undo basis change
    qc.h(0)
    qc.h(1)
    qc.s(1)
    return qc

In [ ]:
def parity_expectation(counts, qubits):
    total = sum(counts.values())
    result = 0.0
    for bits, count in counts.items():
        value = 1
        for q in qubits:
            bit = int(bits[-1-q])
            value *= 1 if bit == 0 else -1
        result += value * count/total
    return result

def sampled_energy(theta_array, backend=None):
    theta = float(np.atleast_1d(theta_array)[0])
    values = {}

    for basis in ["Z", "X", "Y"]:
        qc = ansatz(theta)
        # Basis change for measurement
        if basis == "X":
            qc.h([0, 1])
        elif basis == "Y":
            qc.sdg([0, 1])
            qc.h([0, 1])

        qc.measure_all()
        counts = run_counts(
            qc,
            shots=2048,
            backend=backend
        )
        values[basis] = counts

    # Expectation values
    z0 = parity_expectation(values["Z"], [0])
    z1 = parity_expectation(values["Z"], [1])
    zz = parity_expectation(values["Z"], [0, 1])
    xx = parity_expectation(values["X"], [0, 1])
    yy = parity_expectation(values["Y"], [0, 1])

    # Energy expectation value
    return (
        c0
        + c1*z0
        + c2*z1
        + c3*zz
        + c4*(xx + yy)
    )

## Coefficient sets

The following coefficient sets are a compact data set for the teaching purpose. They illustrate the VQE/geometry workflow; the complete professional workflow would regenerate molecular integrals at each distance using a chemistry driver.

In [ ]:
# R in Angstrom, followed by c0,c1,c2,c3,c4
h2_data = {
    0.50: (-0.72,  0.44, -0.44, -0.02, 0.16),
    0.60: (-0.92,  0.42, -0.42, -0.015, 0.17),
    0.70: (-1.04,  0.40, -0.40, -0.012, 0.18),
    0.74: (-1.0523732458, 0.3979374248, -0.3979374248, -0.0112801043, 0.1809311998),
    0.80: (-1.02,  0.38, -0.38, -0.010, 0.18),
    1.00: (-0.90,  0.32, -0.32, -0.008, 0.17),
    1.40: (-0.72,  0.20, -0.20, -0.004, 0.14),
}

In [ ]:
def hamiltonian_from_coeffs(coeffs):
    c0,c1,c2,c3,c4 = coeffs
    return SparsePauliOp.from_list([
        ("II", c0),
        ("IZ", c1),
        ("ZI", c2),
        ("ZZ", c3),
        ("XX", c4),
        ("YY", c4),
    ])

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Fake IBM Sherbrooke backend
fake_sherbrooke = FakeSherbrooke()

# Simulator using Sherbrooke's device properties and noise
sherbrooke_simulator = AerSimulator.from_backend(fake_sherbrooke)

ideal_curve = {}
sherbrooke_curve = {}

# ------------------------------------------------------------
# Run for each bond distance R
# ------------------------------------------------------------
for R, coeffs in h2_data.items():

    c0, c1, c2, c3, c4 = coeffs

    # ----- Shot-based ideal VQE -----
    ideal_result = minimize(
        sampled_energy,
        x0=[0.0],
        method="COBYLA",
        options={"maxiter": 35}
    )
    ideal_curve[R] = ideal_result.fun

    # ----- FakeSherbrooke VQE -----
    sherbrooke_result = minimize(
        lambda x: sampled_energy(
            x,
            backend=sherbrooke_simulator
        ),
        x0=[0.0],
        method="COBYLA",
        options={"maxiter": 35}
    )
    sherbrooke_curve[R] = sherbrooke_result.fun

    print(
        f"R = {R:.2f} A | "
        f"Ideal = {ideal_curve[R]:.6f} | "
        f"FakeSherbrooke = {sherbrooke_curve[R]:.6f}"
    )

# ------------------------------------------------------------
# Plot potential-energy curves
# ------------------------------------------------------------

R_values = list(ideal_curve.keys())

plt.figure(figsize=(8, 5))

plt.plot(
    R_values,
    [ideal_curve[R] for R in R_values],
    "s--",
    label="Shot-based ideal VQE"
)

plt.plot(
    R_values,
    [sherbrooke_curve[R] for R in R_values],
    "o-",
    label="FakeSherbrooke VQE"
)

plt.xlabel("Bond distance R (Angstrom)")
plt.ylabel("Ground-state energy (Hartree)")
plt.title("H2 Potential-Energy Curve")
plt.legend()
plt.grid(alpha=0.3)

plt.show()

## Reflection
Explain the difference between:
- the inner VQE loop at fixed $R$, and
- the outer geometry loop over $R$.

```python
best_R = min(exact_curve, key=exact_curve.get)
print(best_R, exact_curve[best_R])
```

The inner loop changes circuit parameters to find the lowest electronic energy for one fixed molecular geometry. The outer loop changes the molecular geometry itself and repeats the electronic calculation.